# Spatial discretization
\
There are two spatial methods implemented in CADET-Core: Discontinuous Galerkin (DG) and FInite Volumes (FV).\
**TODO**: add more info, see e.g. [<u>our documentation</u>](https://cadet.github.io/master/interface/spatial_discretization_methods.html).

### QUESTION:
In the setting below, we consider a load-wash-elute example with three components and a salt, modeled by a General Rate Model with Steric Mass-Action Binding.
Change the discretization inputs and investigate errors and computational efficiency.
Note that method "0" is FV, method > 0 is DG with the corresponding polynomial degree.\
The number of discrete points used for FV is Axial cells times (particle cells + 1)\
For DG, it's [(polynomial degree + 1) times axial cells] times [1 + particle cells times (polynomial degree + 1)]

#### What is the most efficient discretization to get a solution with at least .5% accuracy ?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive
import ipywidgets as widgets
import os

from cadet import Cadet
import utility.convergence as convergence
import utility.setting_Col1D_SMA_4comp_LWE_benchmark1 as lwe

Cadet.cadet_path = r"C:\Users\jmbr\OneDrive\Desktop\CADET_compiled\master5_generalizedUnit_f1a1972\aRELEASE"
path = os.getcwd()

def graph_column(spatial_method=3, nAxElem=2, nParElem=1):

    axRefinement = nAxElem / 8
    model = Cadet()
    model.root = lwe.get_model(
        spatial_method_bulk=spatial_method,
        spatial_method_particle=spatial_method,
        particle_type='GENERAL_RATE_PARTICLE',
        axRefinement=axRefinement,
        parZ=nParElem,
        return_bulk=True,
        idas_abstol=1E-8,
        idas_reltol=1E-6
    )
    
    model.filename = 'test.h5'
    model.save()
    model.run_simulation()

    outlet = convergence.get_outlet(path+'/'+model.filename, unit="000")
    sol_time = convergence.get_solution_times(path+'/'+model.filename)
    
    reference = convergence.get_outlet(path+'/data/ref_LWE.h5', unit="000")
    errorComp = np.max(abs(reference[:, 1:] - outlet[:, 1:])) / np.max(abs(reference[:, 1:]))
    errorSalt = np.max(abs(reference[:, 0] - outlet[:, 0])) / np.max(abs(reference[:, 0]))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # First plot
    axs[0].plot(sol_time, outlet[:, 0])
    axs[0].set_xlabel(r'$x~/~M$')
    axs[0].set_ylabel(r'$concentration~/~mol \cdot M^{-3}$')
    axs[0].set_title("Salt")
    
    axs[1].plot(sol_time, outlet[:, 1:])
    axs[1].set_xlabel(r'$x~/~M$')
    axs[1].set_title("components")

    fig.suptitle(
        f"Sim. time: {convergence.get_compute_time(path + '/' + model.filename):.3e}" + f", rel. max. error salt: {errorSalt*100:.2f}%," + f" rel. max. error comps: {errorComp*100:.2f}%",
        fontsize=14
    )
    
    plt.tight_layout()
    plt.show()

spatial_method_options = [0, 1, 2, 3, 4, 5]
nAxElem_options = [4, 8, 16, 32, 64, 128, 256]
nParElem_options = [1, 2, 4, 8, 16]

interact(
    graph_column,
    spatial_method=widgets.SelectionSlider(
        options=spatial_method_options,
        description="Method"
    ),
    nAxElem=widgets.SelectionSlider(
        options=nAxElem_options,
        description="Axial cells"
    ),
    nParElem=widgets.SelectionSlider(
        options=nParElem_options,
        description="Particle cells"
    )
)



interactive(children=(SelectionSlider(description='Method', options=(0, 1, 2, 3, 4, 5), value=0), SelectionSli…

<function __main__.graph_column(spatial_method=3, nAxElem=2, nParElem=1)>

## Discussion: (results)

## Convergence order

What drives the efficiency of numerical methods for smooth settings (i.e. solutions without discontinuous or steep concentration profiles), is the theoretical convergence order of the method.
The experimental order of convergence (EOC) is computed by
$$
EOC = = 
\frac{\log\left(\dfrac{e_n}{e_{n-1}}\right)}
{\log\left(\dfrac{h_n}{h_{n-1}}\right)},
$$
where $e_n$ is some error metric (e.g. maximal absolute error) for the $n$th refinement, and $h_n$ is the corresponding number of degrees of freedom.\
For FV: $h_n = number of cells$\
For DG: $h_n = (polynomial degree + 1) \cdot number of cells$


**TODO:** consider a smooth setting and compute the convergence order for FV and DG with different polynomial degrees

## Smoothness and oscillations

We can consider something non-smooth, e.g. a step pulse injection, or some non-smooth initial condition, and see how different spatial numerical methods deal with that.\
There is a more complicated example for a to-component Langmuir binding that causes steep, self-retaining concentration fronts, see https://github.com/JuBiotech/Supplement-to-Breuer-et-al.-2023a/blob/main/LRM_Langmuir.py


## Note that Spatial and temporal discretization parameters should optimally result in about the same level of accuracy to get the best computational performance: The larger error of spatial/temporal discretization dominates, and destroys the additional computational efforts of the other discretization's lower tolerance